# CSCI 447/547 Hackathon 8: Dimensionality Reduction and PCA

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will use principal component analysis (PCA) to compress data with many features into just a few, first by hand and then with `scikit-learn`, and see how it affects visualization and classification.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon8-PCA_dimensionality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you do not need to download anything: every dataset in this notebook comes with `scikit-learn`. (The faces in the final section are downloaded automatically the first time you run that cell.)

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon8-PCA_dimensionality.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

---

## 1. What Is Dimensionality Reduction?

**Dimensionality reduction** shrinks a dataset with $n$ features down to $k$ features, where $k$ is much smaller than $n$ (written $k \ll n$), while keeping as much of the important information as possible. It helps with:

- **Visualization:** we can't plot more than 3 dimensions, but we can plot a 2D or 3D summary of the data.
- **Redundancy:** many datasets have correlated features that carry overlapping information.
- **Speed and storage:** fewer features take less memory, and less time to train on.
- **Overfitting:** fewer features give a model fewer ways to fit noise. (Remember the question at the end of Hackathon 1: what happens to KNN as we add dozens or hundreds of features?)

<img alt="reducing 3D data to 2D" src="https://miro.medium.com/v2/resize:fit:948/format:webp/1*hM0-FieMKYTU9Inuz7rRXg.jpeg" width=500>

### 1.1 Load the Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid')

from sklearn.datasets import load_iris, load_digits, fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

### 1.2 Look at the Data

We'll start with the Iris dataset from Hackathons 0 and 6: 150 flowers from 3 species (the colors below: purple = setosa, teal = versicolor, yellow = virginica), each described by 4 measurements in centimeters. Even with only 4 features, we can't plot them all at once. Here are 2 of the features, and then 3 of them, with the 4th shown as the size of each point:

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
X.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
y = iris.target # 0 = setosa, 1 = versicolor, 2 = virginica

fig = plt.figure(figsize=(15, 6))
ax = fig.add_subplot(1, 2, 1)
ax.scatter(X['sepal_length'], X['sepal_width'], c=y, cmap='viridis', edgecolors='k')
ax.set(xlabel='sepal length (cm)', ylabel='sepal width (cm)', title='2 Features')

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(X['sepal_length'], X['sepal_width'], X['petal_length'], c=y, cmap='viridis', edgecolors='k',
           s=10 + 25 * X['petal_width']) # the 4th feature sets the size of each point
ax.set(xlabel='sepal length', ylabel='sepal width', zlabel='petal length', title='3 Features (point size = petal width)')
plt.show()

A **pair plot** shows every pair of features side by side, and the table below it shows the correlation between each pair of features. This works for 4 features, but the digits dataset from Hackathon 6 has 64 features: that would be 64 × 64 = 4,096 plots!

In [ ]:
sns.pairplot(X.assign(species=iris.target_names[y]), hue='species', height=2.5)
plt.show()

X.corr().round(2)

### Think-Pair-Share 1: Redundant Features
**Context:** Look at the pair plot and the correlation table above.

1. **Think:** If you could keep only **two** of the four features, which two would you choose to tell the species apart, and why? If you already know a flower's petal length, how much new information does its petal width add?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Instead of choosing two of the original features, could you *combine* the four features into two new ones that keep more of the information? Discuss what such combinations might look like.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 2. How PCA Works

**Principal Component Analysis (PCA)**, invented by Karl Pearson in 1901, is the most widely used dimensionality reduction method. Picture a 2D dataset whose points form a stretched, tilted cloud. PCA looks for the line through the center of the cloud that the data is most *spread out* along:

<img alt="the best-fitting line through 2D data" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*4XZzGobQIdYxmDwF4cfw8A.png" width=450>

To judge a line, PCA **projects** each point onto it (slides each point to the closest spot on the line):

<img alt="projecting points onto a line" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*8HDovvHDzayaKqapGiWsUQ.png" width=450>

The best line is the one where the projected points are the most spread out (they have the largest **variance**). Thanks to the Pythagorean theorem, that's also the line that is closest to the points overall, so it loses the least information. This line is the **first principal component (PC1)**. The second principal component (PC2) is the direction perpendicular to PC1 with the most remaining spread, and so on: $n$ features give $n$ principal components, ordered from the most variance to the least.

In other words, PCA **rotates** the data onto new axes, so that PC1 becomes the horizontal axis:

<img alt="rotating the data so PC1 is horizontal" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*Vr5-JbLzB6gKSRssZ8Kw5w.png" width=450>

Each principal component is a weighted combination of the original features, and its **explained variance ratio** is the fraction of the data's total variance it captures. To reduce the number of dimensions, we keep only the first few components (the ones that capture most of the variance), and drop the rest.

### Think-Pair-Share 2: Units and Variance
**Context:** PCA looks for the directions with the most variance. All four iris measurements are in centimeters.

1. **Think:** Suppose a classmate records petal length in millimeters, but the other three measurements in centimeters. Petal length's numbers become 10 times bigger, so its variance becomes 100 times bigger. What do you think PC1 will look like, and how much of the variance will it explain? Is that a problem?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** How did we fix this kind of problem in earlier hackathons, like KNN and clustering?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class. (You'll check your answer in Section 4.1.)

---

## 3. PCA Under the Hood

PCA starts by standardizing the data. Then it computes the **covariance matrix**, where entry $(i, j)$ measures how much features $i$ and $j$ vary together. Its diagonal holds each feature's variance, and for standardized data, it's the same as the correlation table from Section 1.2:

In [ ]:
Z = StandardScaler().fit_transform(X) # standardize: mean 0, standard deviation 1
cov = np.cov(Z, rowvar=False)          # the 4 x 4 covariance matrix (rowvar=False: each column is a feature)

sns.heatmap(pd.DataFrame(cov, index=X.columns, columns=X.columns), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Covariance Matrix of the Standardized Iris Data')
plt.show()

The principal components are the **eigenvectors** of the covariance matrix, and each one's **eigenvalue** is the variance of the data along that component. (*Why* this works is a linear algebra story for CSCI 457/557. Here, we'll just use it!) NumPy's `np.linalg.eigh()` computes them (handled for you). It returns them from the smallest eigenvalue to the largest, so we flip them into PCA's order:

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(cov)
order = np.argsort(eigenvalues)[::-1] # the positions of the eigenvalues, from largest to smallest
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order] # each COLUMN is one principal component

print('Explained variance ratio:', (eigenvalues / eigenvalues.sum()).round(3))
pd.DataFrame(eigenvectors, index=X.columns, columns=['PC1', 'PC2', 'PC3', 'PC4']).round(2)

PC1 explains 73% of the variance, and PC2 another 23%, so just two components keep about 96% of the information in all four features.

### 3.1 Coding Exercise: Project onto the Principal Components

**Projecting** the data onto the principal components gives each flower's coordinates along the new axes. It's a matrix multiplication of the standardized data `Z` with the eigenvectors you want to keep (`@`, or `.dot()` from Hackathon 2).

**What you need to write:** Create `Z_2d`, the standardized iris data projected onto the first two principal components (the first two columns of `eigenvectors`). Then make a scatter plot of the flowers in this new 2D space, colored by species, with labeled axes. `plt.scatter()` takes a `c` argument for the colors, just like the plots in Section 1.2.

*Expected:* `Z_2d` should have the shape (150, 2). Setosa should sit off on its own along PC1, with versicolor and virginica next to each other but mostly separate. Compare your plot to the 2D plot in Section 1.2: which one separates the species better?

In [ ]:
Z_2d = # --- YOUR CODE HERE ---
print('Z_2d shape:', Z_2d.shape)

# --- YOUR CODE HERE ---

## 4. PCA with scikit-learn

In practice, scikit-learn's `PCA` does all of this for us. (It uses a closely related method called the singular value decomposition, or SVD, which is faster and more accurate.) It works like the other scikit-learn tools: `n_components` sets how many components to keep, `fit()` finds them, and `transform()` (or `fit_transform()`) projects the data. Once it's fit, `explained_variance_ratio_` holds each component's share of the variance, and `components_` holds the components themselves, one row per component (handled for you):

In [ ]:
pca = PCA(n_components=2)
Z_pca = pca.fit_transform(Z)

print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca.components_, index=['PC1', 'PC2'], columns=X.columns).round(2)

These match our results from Section 3, except that PC1's weights may have the opposite signs. That's fine! A line pointing left is the same line as one pointing right, so the sign of a principal component is arbitrary: scikit-learn's plot would just be a mirror image of yours.

### Think-Pair-Share 3: What Do the Components Mean?
**Context:** The table above shows how much each original feature contributes to each principal component. These weights are called **loadings**.

1. **Think:** Which features have large loadings in PC1, and do they push in the same direction? Which feature dominates PC2? Give each component a name, like you named the customer segments in Hackathon 7.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** In your Section 3.1 plot, setosa was separated from the other species along PC1. Using your names for the components, what does that tell you about setosa flowers? What's one downside of describing flowers with PC1 and PC2 instead of the original measurements?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 4.1 Coding Exercise: Checking Think-Pair-Share 2

The cell below makes `X_mm`, a copy of the iris data with petal length in millimeters (handled for you).

**What you need to write:** Fit a 2-component PCA to the **unscaled** `X_mm` the same way as the example above (but without standardizing first), and show its explained variance ratio and its components, labeled with the feature names. (`PCA` always centers each feature at a mean of 0, but it doesn't scale them.) Was your prediction from Think-Pair-Share 2 right?

*Expected:* PC1 explains about 99.9% of the variance, and its loading for petal length is about 1.00.

In [ ]:
X_mm = X.copy()
X_mm['petal_length'] = X_mm['petal_length'] * 10 # centimeters -> millimeters

# --- YOUR CODE HERE ---

## 5. A Bigger Example: Handwritten Digits

Iris only has 4 features. The digits dataset from Hackathon 6 has 64: each 8×8 image has one feature per pixel. As in Hackathon 6, we set aside 25% of the images as a generalization set, and we learn the scaling from the training images only. Here's what the training images look like when PCA squeezes all 64 pixels down to 2 components (handled for you). Each point is one image, colored by its digit.

In [ ]:
digits = load_digits()
X_train, X_gen, y_train, y_gen = train_test_split(digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target)

X_train_scaled = StandardScaler().fit_transform(X_train)

pca = PCA(n_components=2)
digits_2d = pca.fit_transform(X_train_scaled)
print(f'Variance explained by 2 components: {pca.explained_variance_ratio_.sum():.3f}')

plt.figure(figsize=(9, 7))
points = plt.scatter(digits_2d[:, 0], digits_2d[:, 1], c=y_train, cmap='tab10', s=15)
plt.colorbar(points, label='digit')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Digits Compressed from 64 Pixels to 2 Principal Components')
plt.show()

### Think-Pair-Share 4: PCA Never Saw the Labels
**Context:** PCA is **unsupervised**: it only looks at the pixels, never at which digit each image shows. And 2 components keep only about 22% of the variance.

1. **Think:** Even so, many of the digits form their own groups in the plot. Why might the directions with the most variance also tend to separate the digits? Which digits overlap the most?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** PCA keeps the directions with the most variance. But what if the direction that best separates two classes has very little variance? Discuss what could go wrong if you used PCA before a classifier, and how you could check whether it went wrong.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 5.1 Coding Exercise: How Many Components?

2 components are great for plotting, but they throw away almost 80% of the variance. A common rule of thumb is to keep enough components to explain 95% of the variance.

**What you need to write:** Fit a PCA that keeps **all** 64 components (leave out `n_components`) to the scaled training images, and plot the **cumulative** explained variance against the number of components, with a dashed horizontal line at 0.95. Then print how many components it takes to reach 95%. `np.cumsum()` gives running totals (`np.cumsum([0.5, 0.3, 0.2])` gives `[0.5, 0.8, 1.0]`), `plt.axhline()` draws the line (as in Hackathon 7), and `np.argmax()` on a True/False array gives the position of its first `True` (`np.argmax(np.array([0.5, 0.8, 1.0]) >= 0.75)` gives 1). Remember that positions start at 0, so position 1 means 2 components.

*Expected:* 40 of the 64 components explain 95% of the variance.

In [ ]:
# --- YOUR CODE HERE ---

### 5.2 Coding Exercise: Does PCA Hurt a Classifier?

Does dropping components cost accuracy? Here's KNN (from Hackathon 1) on all 64 standardized pixels, scored with 5-fold cross-validation on the training set (handled for you). When we add PCA, it goes *inside* the pipeline, between the scaler and KNN: that way, the scaler and PCA are re-fit on each fold's training data, so nothing leaks from the fold being scored.

In [ ]:
knn_all_pixels = make_pipeline(StandardScaler(), KNeighborsClassifier())
all_pixels_acc = cross_val_score(knn_all_pixels, X_train, y_train, cv=5).mean()
print(f'KNN on all 64 pixels: {all_pixels_acc:.3f}')

**What you need to write:** By the end of the loop, `cv_scores` should hold the cross-validated accuracy of the same KNN pipeline with a PCA step that keeps each number of components in `n_components_list`. Then plot `cv_scores` against `n_components_list`, with a dashed horizontal line at `all_pixels_acc` for comparison, labeled so the legend can tell them apart (`plt.plot()` and `plt.axhline()` both take a `label` argument).

*Expected:* about 0.54 with 2 components, 0.93 with 10, and 0.976 with 30, which matches KNN on all 64 pixels (0.977).

In [ ]:
n_components_list = [2, 5, 10, 20, 30, 40]
cv_scores = []

for n in n_components_list:
    # --- YOUR CODE HERE ---

plt.figure(figsize=(8, 5))
# --- YOUR CODE HERE ---
plt.xlabel('number of components')
plt.ylabel('cross-validated accuracy')
plt.title('KNN Accuracy vs. Number of PCA Components (Digits)')
plt.legend()
plt.show()

With 30 components, KNN is just as accurate as with all 64 pixels, using less than half of the features. Now that we've made our choice using cross-validation, let's train that pipeline on the whole training set and evaluate it on the generalization set (handled for you):

In [ ]:
final_model = make_pipeline(StandardScaler(), PCA(n_components=30), KNeighborsClassifier())
final_model.fit(X_train, y_train)
print(f'Generalization accuracy with 30 components: {final_model.score(X_gen, y_gen):.3f}')

### Advantages and Disadvantages of PCA

- **Advantages:** it removes redundancy between correlated features, it makes models faster to train and less prone to overfitting, and it lets us visualize high-dimensional data.
- **Disadvantages:** the components are harder to interpret than the original features; it only finds straight-line combinations of features, so it can miss curved structure in the data; it keeps the directions with the most variance, which aren't always the most useful for prediction; and keeping too few components throws information away.

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
In Section 5.2, KNN with 30 components did just as well as KNN with all 64 pixels. Describe a situation where you would *not* want to use PCA before a model, even if it doesn't hurt accuracy. What would you be giving up by using it?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate standardizes and runs PCA on their *entire* dataset, then splits it into training and test sets, and reports a suspiciously high test accuracy. What went wrong, and how should they fix it?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario where you would use dimensionality reduction:

    1. 

---

## Extension: Eigenfaces

One of the most famous uses of PCA is **eigenfaces** (Turk & Pentland, 1991). A 64×64 grayscale photo of a face is a vector of 4,096 pixels, so PCA can find the directions that photos of faces vary along the most. Reshaped back into images, those principal components look like ghostly faces, called **eigenfaces**, and every face can be approximated as the average face plus a weighted sum of eigenfaces.

We'll use the Olivetti faces dataset: 400 photos of 40 people. `scikit-learn` downloads it automatically the first time you run the cell below. Every pixel is measured in the same units, so we don't standardize them (PCA still centers each pixel automatically). Here are the average face and the first 8 eigenfaces (handled for you):

In [ ]:
faces = fetch_olivetti_faces().data # 400 faces, each a row of 4,096 pixels
face_pca = PCA(n_components=150).fit(faces)

fig, axs = plt.subplots(nrows=1, ncols=9, figsize=(18, 2.6))
axs[0].imshow(face_pca.mean_.reshape(64, 64), cmap='gray')
axs[0].set_title('average face')
for i, ax in enumerate(axs[1:]):
    ax.imshow(face_pca.components_[i].reshape(64, 64), cmap='gray') # light = positive weight, dark = negative
    ax.set_title(f'eigenface {i + 1}')
for ax in axs:
    ax.axis('off')
plt.show()

To compress a face, we only store its weights on the first few eigenfaces. Here's one face rebuilt from more and more of them (handled for you). `inverse_transform()` turns a face's weights back into pixels.

In [ ]:
face = faces[[0]]                  # the first face, as a 1-row array
weights = face_pca.transform(face) # its weight on each of the 150 eigenfaces

fig, axs = plt.subplots(nrows=1, ncols=6, figsize=(16, 3.2))
for ax, n in zip(axs, [5, 20, 50, 100, 150]):
    kept = weights.copy()
    kept[:, n:] = 0 # keep only the first n eigenfaces
    ax.imshow(face_pca.inverse_transform(kept).reshape(64, 64), cmap='gray')
    ax.set_title(f'{n} eigenfaces\n({face_pca.explained_variance_ratio_[:n].sum():.0%} of variance)')
axs[-1].imshow(face.reshape(64, 64), cmap='gray')
axs[-1].set_title('original\n(4,096 pixels)')
for ax in axs:
    ax.axis('off')
plt.show()

**Pause point:** About 50 eigenfaces (around 1% as many numbers as the 4,096 pixels) are enough for a recognizable face. But look closely at the eigenfaces themselves. Are they faces of real people? Why do some of them look more like lighting patterns than faces? Could you explain to a friend what "0.8 of eigenface 3" means?

In class, we'll look at an alternative to PCA called **archetypal analysis**, whose building blocks are mixtures of *real* faces.